# 01. TCGA Data Exploration

This notebook explores TCGA clinical and molecular data for survival analysis.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.config import RAW_DATA_DIR, DEFAULT_CANCER_TYPE
from src.data.download_tcga import create_sample_data
from src.data.load_tcga import load_tcga_data, validate_survival_data

sns.set_style('whitegrid')
%matplotlib inline

## 1. Create Sample Data

In [ ]:
# Create sample data for demonstration
create_sample_data(DEFAULT_CANCER_TYPE)
print(f"Created sample {DEFAULT_CANCER_TYPE} data")

## 2. Load Data

In [ ]:
# Load clinical and expression data
merged_df, _ = load_tcga_data(
    clinical_path=RAW_DATA_DIR / f"{DEFAULT_CANCER_TYPE}_clinical.csv",
    expression_path=RAW_DATA_DIR / f"{DEFAULT_CANCER_TYPE}_expression.csv",
    merge=True
)

print(f"Data shape: {merged_df.shape}")
merged_df.head()

## 3. Validate Survival Data

In [ ]:
# Validate survival data
validated_df = validate_survival_data(merged_df)
print(f"Validated shape: {validated_df.shape}")

## 4. Exploratory Data Analysis

In [ ]:
# Clinical features distribution
clinical_cols = ['age_at_diagnosis', 'gender', 'tumor_stage']

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Age distribution
axes[0].hist(validated_df['age_at_diagnosis'], bins=20, edgecolor='black')
axes[0].set_xlabel('Age at Diagnosis')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Age Distribution')

# Gender distribution
validated_df['gender'].value_counts().plot(kind='bar', ax=axes[1])
axes[1].set_xlabel('Gender')
axes[1].set_ylabel('Count')
axes[1].set_title('Gender Distribution')
axes[1].tick_params(axis='x', rotation=0)

# Tumor stage distribution
validated_df['tumor_stage'].value_counts().plot(kind='bar', ax=axes[2])
axes[2].set_xlabel('Tumor Stage')
axes[2].set_ylabel('Count')
axes[2].set_title('Tumor Stage Distribution')
axes[2].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## 5. Survival Analysis

In [ ]:
# Survival time distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Overall survival time
axes[0].hist(validated_df['OS_time'], bins=30, edgecolor='black')
axes[0].set_xlabel('Survival Time (days)')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Survival Time Distribution')

# Survival by event status
validated_df[validated_df['OS_status']==1]['OS_time'].hist(
    bins=30, alpha=0.5, label='Event', ax=axes[1], color='red'
)
validated_df[validated_df['OS_status']==0]['OS_time'].hist(
    bins=30, alpha=0.5, label='Censored', ax=axes[1], color='blue'
)
axes[1].set_xlabel('Survival Time (days)')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Survival Time by Event Status')
axes[1].legend()

plt.tight_layout()
plt.show()

## 6. Kaplan-Meier Curves

In [ ]:
from src.evaluation.survival_plots import plot_kaplan_meier

# Overall KM curve
fig = plot_kaplan_meier(
    event_times=validated_df['OS_time'].values,
    event_observed=validated_df['OS_status'].values,
    title='Overall Survival - Kaplan-Meier Curve'
)
plt.show()

In [ ]:
# KM curves by tumor stage
fig = plot_kaplan_meier(
    event_times=validated_df['OS_time'].values,
    event_observed=validated_df['OS_status'].values,
    groups=validated_df['tumor_stage'].values,
    title='Survival by Tumor Stage'
)
plt.show()

## 7. Gene Expression Overview

In [ ]:
# Get gene expression columns
gene_cols = [c for c in validated_df.columns if c.startswith('GENE')]
print(f"Number of genes: {len(gene_cols)}")

# Expression statistics
expression_data = validated_df[gene_cols]
print(f"\nExpression statistics:")
print(expression_data.describe())

In [ ]:
# Heatmap of top variable genes
top_genes = expression_data.var().nlargest(20).index.tolist()

plt.figure(figsize=(10, 8))
sns.heatmap(
    expression_data[top_genes].iloc[:50].T,
    cmap='RdBu_r',
    center=0,
    cbar_kws={'label': 'Expression'}
)
plt.title('Top 20 Variable Genes (First 50 Patients)')
plt.xlabel('Patient')
plt.ylabel('Gene')
plt.tight_layout()
plt.show()

## Summary

- Loaded and validated TCGA data
- Explored clinical features and survival patterns
- Visualized gene expression profiles
- Next: Feature engineering and model building